In [ ]:
# Exo1

from collections import Counter
import re

# Task 1: Open exo1_server.log and read all lines
with open("./files/exo1_server.log", "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"{len(lines)} lines read\n")

# Task 2 & 3: Match verbose pattern, normalize level, store as dict
LOG_PATTERN = re.compile(
    r"""
    ^
    (?P<date>\d{4}-\d{2}-\d{2})\s+        # Date: YYYY-MM-DD
    (?P<time>\d{2}:\d{2}:\d{2})\s+        # Time: HH:MM:SS
    \[(?P<level>[a-zA-Z]+)\]\s+           # Level in brackets (case-insensitive)
    (?P<ip>\S+)\s+-\s+                    # IP address candidate
    (?P<message>.*?)                      # Log message body
    \s*$
    """,
    re.VERBOSE,
)

bad_lines = []
entries = []

for line_no, line in enumerate(lines, start=1):
    match = LOG_PATTERN.match(line)
    if match:
        entry = match.groupdict() # Get a dictionary of the matched groups
        entry["level"] = entry["level"].upper()
        entries.append(entry)
    else:
        bad_lines.append((line_no, line.strip()))

print(f"{len(entries)} entries, {len(bad_lines)} bad line(s): {bad_lines}\n")

# Task 4: Validate IPv4 addresses (0-255, no leading zeroes)
OCTET = r"(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)"
IPV4 = re.compile(rf"^(?:{OCTET}\.){{3}}{OCTET}$")

valid_entries = []
invalid_ips = []

for entry in entries:
    if IPV4.fullmatch(entry["ip"]):
        valid_entries.append(entry)
    else:
        invalid_ips.append(entry["ip"])

invalid_ips_sorted = sorted(invalid_ips)
print(f"Invalid IPs: {invalid_ips_sorted}\n")

# Task 5: Frequency counting using collections.Counter
# Count ERRORs only for valid IP addresses
errors_per_valid_ip = Counter(
    entry["ip"] for entry in valid_entries if entry["level"] == "ERROR" # for each valid entry, if level is ERROR, count the IP
)
print(errors_per_valid_ip, "\n")

# Count total entries per level across all parsed entries
level_counts = Counter(entry["level"] for entry in entries)
print(level_counts, "\n")

# Task 6: Extract identifiers and compute average timeout
# Extract from ALL ERROR entries (including lines with invalid IPs)
error_entries = [entry for entry in entries if entry["level"] == "ERROR"]
all_error_messages = "\n".join(entry["message"] for entry in error_entries)

# Single re.findall capturing (key, value) separately
identifiers = re.findall(
    r"\b([a-zA-Z_]+)=([A-Z]-\d+)\b", all_error_messages
)

# Extract timeout values using re.search
timeout_durations = []
for entry in error_entries:
    timeout_match = re.search(r"after\s+(\d+)s", entry["message"])
    if timeout_match:
        timeout_durations.append(int(timeout_match.group(1)))

avg_timeout = (
    sum(timeout_durations) / len(timeout_durations) if timeout_durations else 0.0
)

# Task 7: Clean report matching exact formatting
print("=== LOG REPORT ===")
print(f"Parsed entries: {len(entries)}")
print(f"Bad lines: {[line_no for line_no, _ in bad_lines]}")
print(f"Entries invalid IP: {len(invalid_ips)} -> {invalid_ips_sorted}")
print("Errors per valid IP:")
for ip, count in errors_per_valid_ip.items():
    print(f"   {ip:<15} {count}")
print(f"Identifiers: {identifiers}")
print(f"Average timeout: {avg_timeout:.1f}s over {len(timeout_durations)} timeouts")

14 lines read

13 entries, 1 bad line(s): [(8, 'corrupted line without proper format')]

Invalid IPs: ['010.0.0.7', '256.1.1.1', '999.12.1.4']

Counter({'192.168.1.23': 2, '10.0.0.5': 2, '172.16.4.8': 1}) 

Counter({'ERROR': 6, 'INFO': 3, 'WARNING': 2, 'DEBUG': 1, 'CRITICAL': 1}) 

=== LOG REPORT ===
Parsed entries: 13
Bad lines: [8]
Entries invalid IP: 3 -> ['010.0.0.7', '256.1.1.1', '999.12.1.4']
Errors per valid IP:
   192.168.1.23    2
   10.0.0.5        2
   172.16.4.8      1
Identifiers: [('query_id', 'Q-4521'), ('query_id', 'Q-4533'), ('code', 'E-107'), ('query_id', 'Q-4602'), ('code', 'E-210'), ('code', 'E-300')]
Average timeout: 29.0s over 3 timeouts


In [15]:
# Exo2

# Task 1: Read the entire content of exo2_contacts.txt
# 1. Read the entire content of exo2_contacts.txt
with open("./files/exo2_contacts.txt", "r", encoding="utf-8") as f:
    full_content = f.read()

# 2. Extract non-empty lines (stripping trailing whitespace/newlines)
non_empty_lines = [
    line.strip() for line in full_content.splitlines() if line.strip() # for line in full_content.splitlines(), strip whitespace, and keep only non-empty lines
]

print("Full content:")
print(full_content)

# Task 2:
NAME = re.compile(r'(?<=)Name: ([A-Za-z \']+)')
names = NAME.findall(full_content)

print("Names extracted:")
print(names)

#Task 3:
MAIL = re.compile(r'(^[a-zA-Z0-9]+([._-][a-zA-Z0-9]+)*$)\@([a-zA-Z0-9]([a-zA-Z0-9-]*[a-zA-Z0-9])?\.)+[a-zA-Z]{2,}$')
emails = MAIL.findall(full_content)

print("Emails extracted:")
print(emails)


Full content:
Name: Amine El Fassi | Email: amine.elfassi@univ-cadi.ac.ma | Tel: +212 6 12 34 56 78
Name: Salma Benali | Email: salma_benali@gmail.com | Tel: 06-61-23-45-67
Name: John Smith | Email: john.smith@@company.com | Tel: 0612345678
Name: Yassine Idrissi | Email: y.idrissi@outlook.fr | Tel: (+212) 661 234 567
Name: Lina O'Brien | Email: lina.obrien@my-startup.io | Tel: 00212 6 98 76 54 32
Name: Bad Entry | Email: bad..dots@example.com | Tel: 12345
Name: Omar Tazi | Email: omar.tazi@example | Tel: +212612345678
Name: Kenza Alaoui | Email: kenza.alaoui@gmail.com | Tel: 07 11 22 33 44
Name: Hamza Berrada | Email: hamza@univ-cadi.ac.ma | Tel: +212 7 00 11 22 33

Names extracted:
['Amine El Fassi ', 'Salma Benali ', 'John Smith ', 'Yassine Idrissi ', "Lina O'Brien ", 'Bad Entry ', 'Omar Tazi ', 'Kenza Alaoui ', 'Hamza Berrada ']
Emails extracted:
[]
